# ABC inference

In [1]:
%run -i ./preamble.py
%config InlineBackend.figure_format = 'retina'


In [2]:
# dill.load_session("Geometric_Exponential.pkl")

In [3]:
import sys
import approxbayescomp as abc
print("Python version:", sys.version)
print("Numpy version:", np.__version__)
print("PyMC3 version:", pm.__version__)
print("Arviz version:", arviz.__version__)

tic()

Python version: 3.8.20 (default, Oct  3 2024, 15:24:27) 
[GCC 11.2.0]
Numpy version: 1.22.3
PyMC3 version: 3.11.4
Arviz version: 0.12.1


In [4]:
FAST = False

if not FAST:
    numIters = 10
    numItersData = 10
    popSize = 1000
    timeout = 1000
else:
    numIters = 4
    numItersData = 6
    popSize = 100
    timeout = 30
smcArgs = {"verbose": True, "numProcs": 40}

# Create a pseudorandom number generator
rg = default_rng(1234)

# Parameters of the true model
freq = "geometric"
sev = "exponential"
thetaTrue = (0.8, 5)

# Setting the time horizon
T = 100

# Simulating the claim data
freqs, sevs = abc.simulate_claim_data(rg, T, freq, sev, thetaTrue)

# Simulating the observed data
psi = abc.Psi("sum")
xData = abc.compute_psi(freqs, sevs, psi)
print("Number of zeros:", np.sum(xData == 0))
print("Mean summands:", np.mean(sevs))

Number of zeros: 21
Mean summands: 5.11128312938337


## Trying to recover both frequency and claim size distributions

In [5]:
params = ("p", "δ")
prior = abc.IndependentUniformPrior([(0, 1), (0, 100)], params)
model = abc.Model(freq, sev, psi, prior)
%time fit = abc.smc(numIters, popSize, xData, model,priors=prior, **smcArgs)

Starting ABC-SMC with population size of 1000 and sample size of 100 (~> 100) on 40 processes.
Finished sampling from prior, eps = 28.25, time = 127.0s / 2.1m, popSize = 1000 -> 500, ESS = 1000 -> 500, # sims = 1000, total # sims = 1000
Finished SMC iteration 1, eps = 14.98, time = 26.0s / 0.4m, popSize = 1507 -> 522, ESS = 1450 -> 500, # sims = 2720, total # sims = 3720
Finished SMC iteration 2, eps = 10.24, time = 0.0s / 0.0m, popSize = 1544 -> 543, ESS = 1337 -> 500, # sims = 3520, total # sims = 7240
Finished SMC iteration 3, eps = 6.48, time = 0.0s / 0.0m, popSize = 1556 -> 536, ESS = 1444 -> 500, # sims = 3840, total # sims = 11080
Finished SMC iteration 4, eps = 4.14, time = 0.0s / 0.0m, popSize = 1546 -> 531, ESS = 1410 -> 500, # sims = 4320, total # sims = 15400
Finished SMC iteration 5, eps = 2.83, time = 0.0s / 0.0m, popSize = 1553 -> 547, ESS = 1404 -> 500, # sims = 5600, total # sims = 21000
Finished SMC iteration 6, eps = 2.21, time = 1.0s / 0.0m, popSize = 1612 -> 594, E

## Trying to recover just the claim sizes conditioned on the observed frequency counts

In [6]:
priorData = abc.IndependentUniformPrior([(0, 100)], ("p", "δ"))
modelData = abc.Model(freqs, sev, psi, priorData)
%time fitData = abc.smc(numItersData, popSize, xData, modelData,priors=priorData, **smcArgs)

Starting ABC-SMC with population size of 1000 and sample size of 100 (~> 100) on 40 processes.
Finished sampling from prior, eps = 186.12, time = 1.0s / 0.0m, popSize = 1000 -> 500, ESS = 1000 -> 500, # sims = 1000, total # sims = 1000
Finished SMC iteration 1, eps = 49.07, time = 18.0s / 0.3m, popSize = 1559 -> 506, ESS = 1538 -> 500, # sims = 1560, total # sims = 2560
Finished SMC iteration 2, eps = 10.43, time = 0.0s / 0.0m, popSize = 1523 -> 504, ESS = 1497 -> 500, # sims = 1560, total # sims = 4120
Finished SMC iteration 3, eps = 3.40, time = 0.0s / 0.0m, popSize = 1508 -> 503, ESS = 1478 -> 500, # sims = 1560, total # sims = 5680
Finished SMC iteration 4, eps = 2.14, time = 0.0s / 0.0m, popSize = 1550 -> 506, ESS = 1507 -> 500, # sims = 1840, total # sims = 7520
Finished SMC iteration 5, eps = 1.75, time = 0.0s / 0.0m, popSize = 1551 -> 529, ESS = 1493 -> 500, # sims = 3920, total # sims = 11440
Finished SMC iteration 6, eps = 1.51, time = 1.0s / 0.0m, popSize = 1583 -> 544, ESS 

In [ ]:
abc.weighted_distplot(fit.samples[:, 1], fit.weights)
abc.weighted_distplot(fitData.samples[:, 0], fitData.weights)
plt.axvline(thetaTrue[1], c="k", ls="--")
plt.yticks([])
sns.despine(left=True)
plt.legend(["Observing sums", "Observing freqs & sums"], frameon=False)

## Using summary statistics

In [ ]:
from numba import njit


@njit(nogil=True)
def sumstats(x):
    return np.sum(x)


@njit(nogil=True)
def l1_dist(ssData, ssFake):
    return np.abs(ssData - ssFake)

In [ ]:
%%time

fitSS = abc.smc(
    numIters,
    popSize,
    xData,
    model,
    priors=prior,
    sumstats=sumstats,
    distance=l1_dist,
    **smcArgs
)

In [ ]:
basic_model = pm.Model()

with basic_model:
    # Priors for unknown model parameters
    p = pm.Uniform("p", 0, 1)
    δ = pm.Uniform("δ", 0, 100)

    # Log probability of the compound-sum variable
    def logp(t0, sumData):
        return T * np.log(1 - p) + (T - t0) * np.log(p / δ) - (1 - p) / δ * sumData

    exp_surv = pm.DensityDist(
        "X", logp, observed={"t0": np.sum(xData == 0), "sumData": np.sum(xData)}
    )
    
    %time trace = pm.sample(1000, tune=500, chains = 1, random_seed = 1, return_inferencedata=True)

## MLE

In [ ]:
t0 = np.sum(xData == 0)
sumData = np.sum(xData)


def logp(θ):
    p, δ = θ
    return T * np.log(1 - p) + (T - t0) * np.log(p / δ) - (1 - p) / δ * sumData


from scipy.optimize import minimize


def costFn(θ):
    return -logp(θ)


bnds = ((0, 1), (0, None))
x0 = np.array((0.5, 6.0))

minRes = minimize(costFn, x0, bounds=bnds)
print("Optimiser reports a success?", minRes.success)

print(minRes.x, thetaTrue)

pPartialMLE = minRes.x[0]
δPartialMLE = minRes.x[1]

## MLE using full information

In [ ]:
pMLE = 1 - 1 / (freqs.mean() + 1)
δMLE = sevs.mean()

## Plot all together

In [ ]:
fig, axs = plt.subplots(1, len(params), tight_layout=True)
nMCMC = len(trace.posterior["p"][0])

for l, param in enumerate(params):
    pLims = [prior.marginals[l].isf(1), prior.marginals[l].isf(0)]

    abc.weighted_distplot(fit.samples[:, l], fit.weights, ax=axs[l], hist=False)
    abc.weighted_distplot(fitSS.samples[:, l], fitSS.weights, ax=axs[l], hist=False)
    abc.weighted_distplot(trace.posterior[param][0], np.ones(nMCMC) / nMCMC, ax=axs[l], hist=False)
    axs[l].legend([], frameon=False)
    axs[l].set_title("$" + param + "$")
    axs[l].set_yticks([])
    axs[l].axvline(thetaTrue[l], color="k", linestyle="dashed", alpha=0.8)

sns.despine(left=True)
# save_cropped("../Figures/geometric-exponential-posterior.pdf")
fig, axs = plt.subplots(1, len(params), tight_layout=True)

for l, param in enumerate(params):
    pLims = [prior.marginals[l].isf(1), prior.marginals[l].isf(0)]

    abc.weighted_distplot(fit.samples[:, l], fit.weights, ax=axs[l], hist=False)
    abc.weighted_distplot(fitSS.samples[:, l], fitSS.weights, ax=axs[l], hist=False)
    abc.weighted_distplot(trace.posterior[param][0], np.ones(nMCMC) / nMCMC, ax=axs[l], hist=False)
    axs[l].legend([], frameon=False)
    # axs[l].set_title("$" + param + "$")
    axs[l].set_yticks([])
    axs[l].axvline(thetaTrue[l], color="k", linestyle="dashed", alpha=0.8)

draw_prior(prior, axs)
sns.despine(left=True)
save_cropped("../Figures/geometric-exponential-posterior-with-prior.pdf")

In [ ]:
elapsed = toc()
print(f"Notebook time = {elapsed:.0f} secs = {elapsed/60:.2f} mins")

In [ ]:
dill.dump_session("Geometric_Exponential.pkl")